In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer


In [ ]:
dataset = load_dataset("glue", "mnli")
print(dataset["train"][0])
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
def tokenize_fn(batch):
    return tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation = True,
        padding = "max_length",
        max_length = 128
    )
tokenized = dataset.map(tokenize_fn, batched = True)
print(tokenized["train"][0].keys())


In [ ]:
print(tokenized["train"][0])

In [ ]:
tokenized = tokenized.remove_columns(["premise", "hypothesis", "idx"])
tokenized.set_format("torch")
print(tokenized["train"][0].keys())
print(tokenized["train"][0])

In [ ]:
import evaluate
import numpy as np

from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    accuracy = accuracy_metric.compute(predictions=preds, references=labels)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")
    precision = precision_metric.compute(predictions=preds, references=labels, average="macro")
    recall = recall_metric.compute(predictions=preds, references=labels, average="macro")
    return {
        "accuracy": accuracy["accuracy"],
        "f1_macro": f1["f1"],
        "precision_macro": precision["precision"],
        "recall_macro": recall["recall"],
    }

In [ ]:
small_train = tokenized["train"].select(range(40000))
small_val = tokenized["validation_matched"].select(range(2000))

In [ ]:
from transformers import TrainingArguments

training_args  = TrainingArguments(
    output_dir="./bert_mnli",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = small_train,
    eval_dataset = small_val,
    compute_metrics = compute_metrics,
)

trainer.train()

In [ ]:
# Experiment 2: lr = 1e-5

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. New model from scratch (pretrained BERT base, not your already fine-tuned model)
model_exp2 = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

# 2. New training arguments
training_args_exp2 = TrainingArguments(
    output_dir="./bert_mnli_exp2_lr1e5",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    learning_rate=1e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none"
)

# 3. New trainer
trainer_exp2 = Trainer(
    model=model_exp2,
    args=training_args_exp2,
    train_dataset=small_train,
    eval_dataset=small_val,
    compute_metrics=compute_metrics,
)

# 4. Train
trainer_exp2.train()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# 1. Получаем предсказания на validation
pred_output = trainer.predict(small_val)
pred_output_exp2=trainer_exp2.predict(small_val)
# 2. Превращаем logits в классы
preds = np.argmax(pred_output.predictions, axis=-1)
labels = pred_output.label_ids

preds_exp2 = np.argmax(pred_output_exp2.predictions, axis=-1)
labels_exp2 = pred_output_exp2.label_ids
# 3. Печатаем метрики от trainer.predict
print("Predict metrics for exp1:")
print(pred_output.metrics)
print()
print("Predict metrics for exp2:")
print(pred_output_exp2.metrics)

# 4. Подробный отчет по классам
print("\nClassification report 1")
print(classification_report(
    labels,
    preds,
    digits=4
))
print("\nClassification report 2:")
print(classification_report(
    labels_exp2,
    preds_exp2,
    digits=4
))

# 5. Матрица ошибок
cm = confusion_matrix(labels, preds)
cm2 = confusion_matrix(labels_exp2, preds_exp2)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["contradiction", "entailment", "neutral"]
)

disp2 = ConfusionMatrixDisplay(
    confusion_matrix=cm2,
    display_labels=["contradiction", "entailment", "neutral"]
)

disp.plot()
disp2.plot()
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Experiment 3: lr = 3e-5

# 1. New model from scratch (pretrained BERT base, not your already fine-tuned model)
model_exp3 = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

# 2. New training arguments
training_args_exp3 = TrainingArguments(
    output_dir="./bert_mnli_exp2_lr1e5",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    learning_rate=3e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none"
)

# 3. New trainer
trainer_exp3 = Trainer(
    model=model_exp3,
    args=training_args_exp3,
    train_dataset=small_train,
    eval_dataset=small_val,
    compute_metrics=compute_metrics,
)

# 4. Train
trainer_exp3.train()

In [ ]:
 # Experiment 4: dataset 40000

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. New model from scratch (pretrained BERT base, not your already fine-tuned model)
model_exp4 = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

# 2. New training arguments
training_args_exp4 = TrainingArguments(
    output_dir="./bert_mnli_exp4_traindata_40000",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none"
)

# 3. New trainer
trainer_exp4 = Trainer(
    model=model_exp4,
    args=training_args_exp4,
    train_dataset=small_train,
    eval_dataset=small_val,
    compute_metrics=compute_metrics,
)

# 4. Train
trainer_exp4.train()

In [ ]:
я# Experiment 5: dataset 40000, batch = 4

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. New model from scratch (pretrained BERT base, not your already fine-tuned model)
model_exp5 = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

# 2. New training arguments
training_args_exp5 = TrainingArguments(
    output_dir="./bert_mnli_exp5_traindata_40000_batch_4",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none"
)

# 3. New trainer
trainer_exp5 = Trainer(
    model=model_exp5,
    args=training_args_exp5,
    train_dataset=small_train,
    eval_dataset=small_val,
    compute_metrics=compute_metrics,
)

# 4. Train
trainer_exp5.train()